# 가중치 민감도 검증 — 30/30/20/20을 흔들면 판정이 얼마나 바뀌나

산식 가중치(주행거리 30 / 생활권 안 안전 30 / 밖 안전 20 / 패턴 안정성 20)를
바꿔가며 180개 시나리오의 **연간 우대 판정**을 다시 계산한다.

- 케어 판정은 가중치를 쓰지 않으므로(변화 지수 기반) 영향이 없다 — 우대 축만 재판정한다.
- 월별 축 점수는 시뮬레이션 번들에 저장된 값을 그대로 쓰고, 가중치만 바꾼다.
- 우대 규칙은 엔진과 동일: 통합점수 75점 이상, 평가 12개월 중 9개월 충족, 커버리지 80% 미만은 보류.

**주의**: 합성 시나리오 기반이므로 "이 값이 옳다"의 증거가 아니다.
값을 흔들었을 때 결과가 얼마나 움직이는지(둔감성)를 보는 용도다.

In [ ]:
# 데이터 로드 — 로컬 저장소가 있으면 그 파일을, 없으면(Colab) GitHub에서 받는다
import json, os, urllib.request
from pathlib import Path

LOCAL = Path("data/fixtures/gaip_simulation_bundle.json")
RAW_URL = ("https://raw.githubusercontent.com/summit1123/seniorcareservice/"
           "claude/gaip-dashboard-refine/data/fixtures/gaip_simulation_bundle.json")

if LOCAL.exists():
    bundle = json.loads(LOCAL.read_text(encoding="utf-8"))
elif Path("../data/fixtures/gaip_simulation_bundle.json").exists():
    bundle = json.loads(Path("../data/fixtures/gaip_simulation_bundle.json").read_text(encoding="utf-8"))
else:
    print("로컬 파일이 없어 GitHub에서 내려받습니다...")
    with urllib.request.urlopen(RAW_URL) as r:
        bundle = json.loads(r.read().decode("utf-8"))

drivers = bundle["drivers"]
print(f"시나리오 {len(drivers)}건 로드 완료")

In [ ]:
# 재판정 함수 — 엔진의 우대 규칙을 그대로 재현
AXES = ("mileage_score", "in_zone_safe_score", "out_zone_safe_score", "pattern_stability_score")
REWARD_THRESHOLD = 75.0
REWARD_REQUIRED_MONTHS = 9
MIN_COVERAGE_PCT = 80.0

def annual_reward_states(drivers, weights):
    w = dict(zip(AXES, weights))
    states = []
    for d in drivers:
        reward_months = eligible_months = 0
        for m in d["monthly_results"]:
            if m["period_role"] != "evaluation":
                continue
            if not m.get("zone_available") or float(m.get("data_coverage_pct", 0)) < MIN_COVERAGE_PCT:
                continue  # 보류 — 판정 미진입
            observed = [(float(m[a]), w[a]) for a in AXES if m.get(a) is not None]
            ow = sum(wt for _, wt in observed)
            if ow <= 0:
                continue
            eligible_months += 1
            score = sum(v * wt for v, wt in observed) / ow  # 미관측 축 재정규화
            if score >= REWARD_THRESHOLD:
                reward_months += 1
        if eligible_months < REWARD_REQUIRED_MONTHS:
            states.append("hold")
        elif reward_months >= REWARD_REQUIRED_MONTHS:
            states.append("reward")
        else:
            states.append("neutral")
    return states

In [ ]:
# 가중치 변형별 재판정
VARIANTS = {
    "30/30/20/20 (현행)": (30, 30, 20, 20),
    "30/30/15/25":        (30, 30, 15, 25),
    "30/30/25/15":        (30, 30, 25, 15),
    "25/35/20/20":        (25, 35, 20, 20),
    "25/25/25/25 (균등)": (25, 25, 25, 25),
    "35/25/20/20":        (35, 25, 20, 20),
    "20/40/20/20":        (20, 40, 20, 20),
    "40/30/15/15":        (40, 30, 15, 15),
}

base = annual_reward_states(drivers, VARIANTS["30/30/20/20 (현행)"])
print(f"현행: 우대 {base.count('reward')} / 중립 {base.count('neutral')} / 보류 {base.count('hold')}\n")

results = {}
print(f"{'가중치':22s} {'우대':>4s} {'현행 대비 변경':>10s}")
print("-" * 44)
for name, weights in VARIANTS.items():
    states = annual_reward_states(drivers, weights)
    changed = sum(1 for a, b in zip(base, states) if a != b)
    results[name] = changed
    print(f"{name:22s} {states.count('reward'):4d} {changed:9d}건")

In [ ]:
# 전체 격자 스윕 — 손으로 고른 변형이 아니라 가중치 공간 전체를 훑는다
import itertools, statistics as st
from collections import defaultdict

grid = [c for c in itertools.product(range(5, 90, 5), repeat=4) if sum(c) == 100]
changes = {}
for w in grid:
    s = annual_reward_states(drivers, w)
    changes[w] = sum(1 for a, b in zip(base, s) if a != b)

vals = list(changes.values())
print(f"전체 격자 {len(grid)}개 조합 — 변경 중앙값 {st.median(vals):.0f}건 · 최대 {max(vals)}건")

near = [w for w in grid if all(abs(a-b) <= 5 for a, b in zip(w, (30,30,20,20)))]
nv = [changes[w] for w in near]
print(f"현행 ±5 이웃 {len(near)}개 — 중앙값 {st.median(nv):.0f}건 · 최대 {max(nv)}건")

# 축별 효과 — 주행거리는 그대로 묶고, 안 안전은 주행거리≤30으로 고정해 교란 제거
by_mileage = defaultdict(list)
for w, c in changes.items():
    by_mileage[w[0]].append(c)

by_inzone_cond = defaultdict(list)
for w, c in changes.items():
    if w[0] <= 30:                      # 주행거리 안정 구간으로 고정
        by_inzone_cond[w[1]].append(c)

print("\n주행거리 가중치별 평균 변경 (전체):")
for k in sorted(by_mileage):
    if k <= 50: print(f"  mileage={k:2d}: {st.mean(by_mileage[k]):5.1f}건")
print("\n안 안전 가중치별 평균 변경 (주행거리≤30 조합만):")
for k in sorted(by_inzone_cond):
    if k <= 50: print(f"  in_zone={k:2d}: {st.mean(by_inzone_cond[k]):5.1f}건")

In [ ]:
# 시각화 — 판정을 지배하는 두 결정
# Colab에는 한글 폰트가 없어 축 이름이 빈 네모로 나온다 → 나눔폰트 설치, 실패 시 영문 라벨
import matplotlib, matplotlib.pyplot as plt

def korean_font():
    have = {f.name for f in matplotlib.font_manager.fontManager.ttflist}
    for c in ("AppleGothic", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
        if c in have:
            return c
    try:  # Colab
        import subprocess
        subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum"],
                       check=True, capture_output=True)
        matplotlib.font_manager.fontManager.addfont(
            "/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
        return "NanumGothic"
    except Exception:
        return None

font = korean_font()
if font:
    plt.rcParams["font.family"] = font
plt.rcParams["axes.unicode_minus"] = False
ko = font is not None

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

ks = [k for k in sorted(by_mileage) if k <= 50]
axes[0].bar([str(k) for k in ks], [st.mean(by_mileage[k]) for k in ks],
            color=["#0f766e" if k <= 30 else "#d97706" for k in ks])
axes[0].set_title("주행거리 축 — 키우면 급증 (35부터)" if ko
                  else "Mileage weight: rises sharply from 35")
axes[0].set_xlabel("주행거리 축 가중치" if ko else "mileage weight")
axes[0].set_ylabel("평균 판정 변경 (건/180)" if ko else "avg. verdict changes (of 180)")

ks2 = [k for k in sorted(by_inzone_cond) if k <= 50]
axes[1].bar([str(k) for k in ks2], [st.mean(by_inzone_cond[k]) for k in ks2],
            color=["#d97706" if k < 25 else "#0f766e" for k in ks2])
axes[1].set_title("안 안전 축 — 줄이면 급증, 현행 30이 최저점" if ko
                  else "In-zone weight: rises when reduced; 30 is the minimum")
axes[1].set_xlabel("안 안전 축 가중치 (주행거리≤30 조합만)" if ko
                   else "in-zone weight (mileage<=30 only)")

plt.tight_layout()
plt.show()

## 해석

- **현행 값 주변(±5 이웃 19개 전부)에서는 변경 중앙값 4건, 최대 12건.** 정확한 값
  논쟁(30이냐 28이냐)은 결과를 크게 바꾸지 않는다.
- **전체 공간은 민감하다** — 969개 조합에서 중앙값 23건, 최대 88건. "아무 가중치나
  된다"는 주장이 아니다.
- **판정을 지배하는 결정은 둘이다.**
  1. **주행거리는 키우면 안 된다** — 가중치 30 이하에서는 평평하다가 35부터
     급증한다(50에서 평균 47건). 적게 몰수록 유리한 축이라, 키우면 활동적인
     안전 운전자가 우대에서 탈락한다.
  2. **안 안전은 줄이면 안 된다** — 주행거리≤30으로 고정하고 보면 안 안전
     가중치별 변경이 U자를 그리며, **현행 30이 바닥이다**(6.8건). 5로 줄이면
     30건까지 치솟는다. 행동 증거가 가장 많이 쌓이는 축이라, 줄이면 판정의
     기반이 흔들린다.
- 밖 안전·패턴의 배분은 상대적으로 완만하다. 두 안정 구간(주행거리≤30,
  안 안전≥25)을 지키는 371개 조합에서는 변경이 중앙값 13건, 최대 24건이다.
- **한계**: 합성 시나리오 기반이므로 "이 값이 옳다"의 증거가 아니다. 실측 손해율이
  쌓이면 재보정한다(가중치는 전부 선언적 설정값).